In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Input, Concatenate, Flatten, Dense, UpSampling2D

In [4]:
def inception_module(x, f1, f3_in, f3_out, f5_in, f5_out, f_pool):
    # Path 1
    path1 = Conv2D(f1, (1,1), padding='same', activation='relu')(x)

    # Path 2
    path2 = Conv2D(f3_in, (1,1), padding='same', activation='relu')(x)
    path2 = Conv2D(f3_out, (3,3), padding='same', activation='relu')(path2)

    # Path 3
    path3 = Conv2D(f5_in, (1,1), padding='same', activation='relu')(x)
    path3 = Conv2D(f5_out, (5,5), padding='same', activation='relu')(path3)

    # Path 4
    path4 = MaxPooling2D((3,3), strides=(1,1), padding='same')(x)
    path4 = Conv2D(f_pool, (1,1), padding='same', activation='relu')(path4)

    output = Concatenate(axis=-1)([path1, path2, path3, path4])
    return output

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]
input_layer = Input(shape=(28, 28, 1))
x = inception_module(input_layer, f1=64, f3_in=96, f3_out=128, f5_in=16, f5_out=32, f_pool=32)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)

output_layer = Dense(10, activation='softmax')(x)
model = Model(inputs=input_layer, outputs=output_layer)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 26s 23ms/step - accuracy: 0.8558 - loss: 0.4159 - val_accuracy: 0.8798 - val_loss: 0.3269
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9110 - loss: 0.2420 - val_accuracy: 0.8983 - val_loss: 0.2818
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 15s 17ms/step - accuracy: 0.9351 - loss: 0.1780 - val_accuracy: 0.9005 - val_loss: 0.2918
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9522 - loss: 0.1305 - val_accuracy: 0.9063 - val_loss: 0.3050
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.9642 - loss: 0.0973 - val_accuracy: 0.9012 - val_loss: 0.3449
222/313 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8974 - loss: 0.3720

# 2- Using transfere learning

In [9]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]
input_layer = Input(shape=(28, 28, 1))

#  Convert grayscale to 3 channels
x = Concatenate(axis=-1)([input_layer, input_layer, input_layer])

# Scale up from 28x28 to 84x84
x = UpSampling2D(size=(3, 3))(x)

basemodel = InceptionV3(include_top=False, weights='imagenet', input_tensor=x)
basemodel.trainable = False

x = basemodel.output

x = Flatten()(x)
x = Dense(256, activation='relu')(x)
output_layer = Dense(10, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test))
model.evaluate(x_test, y_test)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 57s 22ms/step - accuracy: 0.8116 - loss: 0.5350 - val_accuracy: 0.8196 - val_loss: 0.4926
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - accuracy: 0.8512 - loss: 0.4040 - val_accuracy: 0.8466 - val_loss: 0.4357
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 15ms/step - accuracy: 0.8702 - loss: 0.3492 - val_accuracy: 0.8389 - val_loss: 0.4671
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - accuracy: 0.8873 - loss: 0.3027 - val_accuracy: 0.8436 - val_loss: 0.4564
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 15ms/step - accuracy: 0.9031 - loss: 0.2604 - val_accuracy: 0.8427 - val_loss: 0.4821
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 15ms/step - accuracy: 0.9175 - loss: 0.2211 - val_accuracy: 0.8412 - val_loss: 0.5344
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - accuracy: 0.9286 - loss: 0.1894 - val_accuracy: 0.8361 - val_loss: 0.5833
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 15ms/step - accuracy: 0.9405 -

[0.6883553266525269, 0.8342999815940857]